## Build a Retrieval Augmented Generation (RAG) App

One of the most powerful applications enabled by LLMs is sophisticated question-answering (Q&A) chatbots. These are applications that can answer questions about specific source information. These applications use a technique known as Retrieval Augmented Generation, or RAG.

This tutorial will show how to build a simple Q&A application over a text data source. Along the way we’ll go over a typical Q&A architecture and highlight additional resources for more advanced Q&A techniques. We’ll also see how LangSmith can help us trace and understand our application. LangSmith will become increasingly helpful as our application grows in complexity.

In [1]:
%pip install --upgrade langchain-text-splitters langchain-community langgraph



Note: you may need to restart the kernel to use updated packages.


LangSmith
Many of the applications you build with LangChain will contain multiple steps with multiple invocations of LLM calls. As these applications get more and more complex, it becomes crucial to be able to inspect what exactly is going on inside your chain or agent. The best way to do this is with LangSmith.

After you sign up at the link above, make sure to set your environment variables to start logging traces:



In [2]:
import getpass
import os
import os
from dotenv import load_dotenv
load_dotenv()

LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"


In [3]:
!pip install -qU langchain-openai

In [4]:
# import os
# from dotenv import load_dotenv
# load_dotenv()
# from langchain_openai import AzureChatOpenAI

# AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
# AZURE_OPENAI_ENDPOINT=os.getenv("AZURE_OPENAI_ENDPOINT")
# AZURE_OPENAI_DEPLOYMENT_NAME=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
# AZURE_OPENAI_API_VERSION=os.getenv("AZURE_OPENAI_API_VERSION")
# os.environ["AZURE_OPENAI_API_KEY"] = AZURE_OPENAI_API_KEY
# llm = AzureChatOpenAI(
#     azure_endpoint=AZURE_OPENAI_ENDPOINT,
#     azure_deployment=AZURE_OPENAI_DEPLOYMENT_NAME,
#     openai_api_version=AZURE_OPENAI_API_VERSION,
# )



In [5]:
!pip install langchain-google-genai

In [17]:
import os
from dotenv import load_dotenv
load_dotenv()

gemini_api_key = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = gemini_api_key
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-pro",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [7]:
print(llm.invoke("hello"))

content='Hello there! How can I help you today?' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []} id='run-11f71e4d-898f-4c2a-913d-ccdb53e0e639-0' usage_metadata={'input_tokens': 2, 'output_tokens': 11, 'total_tokens': 13, 'input_token_details': {'cache_read': 0}}


In [8]:
!pip install -qU langchain-chroma

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.48.1 requires tokenizers<0.22,>=0.21, but you have tokenizers 0.20.3 which is incompatible.


In [9]:
!pip install -qU pypdf langchain_community



In [10]:
from langchain_community.document_loaders import PyPDFLoader
file_path = "/Users/macintosh/TA-DOCUMENT/StudyZone/FPT_WORK/IVY Training/IVY DEV/DailyReport/DL-3-21.1.2024/data/nike_dataset.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))



106


In [11]:
!pip install -qU langchain-huggingface

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chromadb 0.5.23 requires tokenizers<=0.20.3,>=0.13.2, but you have tokenizers 0.21.0 which is incompatible.


In [12]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

In [13]:
from langchain_chroma import Chroma

vector_store = Chroma(embedding_function=embeddings)

In [14]:
!pip install bs4

In [24]:
print(docs[10].page_content[:500])



Our DE&I focus extends beyond our workforce and includes our communities, which we support in a number of ways. We have 
committed to investments that aim to address racial inequality and improve diversity and representation in our communities. We 
also are leveraging our global scale to accelerate business diversity, including investing in business training programs for women 
and increasing the proportion of services supplied by minority-owned businesses.
COMPENSATION AND BENEFITS 
NIKE's tota


In [30]:
print(f"Total characters: {len(docs[70].page_content)}")

Total characters: 1094


In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 501 sub-documents.


In [32]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['f42b11b4-9478-457c-9d7f-e53e0b91f6cd', 'a8db2e9c-eebf-4f49-94f9-6dd2b3fd693d', '78c42b5b-44a6-48bf-a7a9-dc021568bfd1']


In [34]:
retriever = vector_store.as_retriever()



In [35]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)


question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

results = rag_chain.invoke({"input": "What was Nike's revenue in 2023?"})

results

{'input': "What was Nike's revenue in 2023?",
 'context': [Document(id='a9e73769-64d8-4807-bd86-45d24f772c6e', metadata={'page': 89, 'page_label': '90', 'source': '/Users/macintosh/TA-DOCUMENT/StudyZone/FPT_WORK/IVY Training/IVY DEV/DailyReport/DL-3-21.1.2024/data/nike_dataset.pdf'}, page_content='YEAR ENDED MAY 31,\n(Dollars in millions) 2023 2022 2021\nREVENUES\nNorth America $ 21,608 $ 18,353 $ 17,179 \nEurope, Middle East & Africa  13,418  12,479  11,456 \nGreater China  7,248  7,547  8,290 \nAsia Pacific & Latin America  6,431  5,955  5,343 \nGlobal Brand Divisions  58  102  25 \nTotal NIKE Brand  48,763  44,436  42,293 \nConverse  2,427  2,346  2,205 \nCorporate  27  (72)  40 \nTOTAL NIKE, INC. REVENUES $ 51,217 $ 46,710 $ 44,538 \nEARNINGS BEFORE INTEREST AND TAXES\nNorth America $ 5,454 $ 5,114 $ 5,089 \nEurope, Middle East & Africa  3,531  3,293  2,435 \nGreater China  2,283  2,365  3,243 \nAsia Pacific & Latin America  1,932  1,896  1,530 \nGlobal Brand Divisions  (4,841)  (4